# Exploratory Data Analysis — K8s Metrics
Validate collected metrics and confirm anomalies are visible in the data.

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)

In [ ]:
# Load all collected CSVs
files = glob.glob("../data/raw/metrics_*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"]) - pd.Timedelta(hours=5)
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Total rows: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")
df.head()

In [ ]:
# Plot CPU usage — anomalies should be visible as spikes
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for pod, group in df.groupby("pod"):
    axes[0].plot(group["timestamp"], group["cpu_usage_rate"], label=pod, alpha=0.8)
axes[0].set_title("CPU usage rate by pod")
axes[0].set_ylabel("CPU cores/s")
axes[0].legend(fontsize=8)

for pod, group in df.groupby("pod"):
    axes[1].plot(
        group["timestamp"],
        group["memory_working_set_bytes"] / 1e6,
        label=pod,
        alpha=0.8,
    )
axes[1].set_title("Memory working set by pod (MB)")
axes[1].set_ylabel("MB")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../docs/eda_cpu_memory.png", dpi=150)
plt.show()

In [ ]:
# Feature correlation heatmap
numeric_cols = df.select_dtypes(include="number").columns
corr = df[numeric_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.savefig("../docs/eda_correlation.png", dpi=150)
plt.show()

In [ ]:
# Summary statistics by label
feature_cols = [c for c in numeric_cols if c not in ["pod_ready"]]
df.groupby("label")[feature_cols].describe().round(4)